# Fine-Tuning OpenAI Models with Yggdrasil Attack Traces

This notebook demonstrates how to use attack traces generated by Project Yggdrasil to fine-tune OpenAI models for security analysis.

## Prerequisites

- OpenAI API key
- Attack traces generated from Yggdrasil
- Python 3.8+

## Setup

In [ ]:
# Install required packages
!pip install openai pandas numpy matplotlib tiktoken

In [ ]:
import json
import os
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
from openai import OpenAI
import tiktoken

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Dependencies loaded")

## Step 1: Load Attack Traces

Load all attack trace files from Yggdrasil's log directory.

In [ ]:
# Path to attack traces
TRACE_DIR = Path('../../logs/attack-traces')

def load_attack_traces(trace_dir):
    """Load all JSONL attack trace files"""
    traces = []
    
    if not trace_dir.exists():
        print(f"⚠️  Trace directory not found: {trace_dir}")
        print("   Make sure you've run Yggdrasil and generated attack traces first!")
        return traces
    
    for service_dir in trace_dir.iterdir():
        if not service_dir.is_dir():
            continue
            
        for trace_file in service_dir.glob('*.jsonl'):
            with open(trace_file) as f:
                for line in f:
                    if line.strip():
                        try:
                            trace = json.loads(line)
                            traces.append(trace)
                        except json.JSONDecodeError:
                            continue
    
    return traces

traces = load_attack_traces(TRACE_DIR)
print(f"📊 Loaded {len(traces)} attack traces")

## Step 2: Analyze Dataset

Explore the dataset to understand its composition.

In [ ]:
def analyze_traces(traces):
    """Analyze attack trace dataset"""
    stats = {
        'total': len(traces),
        'successful_exploits': 0,
        'failed_attempts': 0,
        'by_realm': defaultdict(int),
        'by_cwe': defaultdict(int),
        'by_event_type': defaultdict(int)
    }
    
    for trace in traces:
        metadata = trace.get('metadata', {})
        
        if metadata.get('exploit_successful'):
            stats['successful_exploits'] += 1
        else:
            stats['failed_attempts'] += 1
        
        realm = metadata.get('realm')
        if realm:
            stats['by_realm'][realm] += 1
        
        cwe = metadata.get('cwe')
        if cwe:
            stats['by_cwe'][cwe] += 1
        
        event_type = metadata.get('event_type')
        if event_type:
            stats['by_event_type'][event_type] += 1
    
    return stats

if traces:
    stats = analyze_traces(traces)
    
    print("📈 Dataset Statistics:")
    print(f"   Total traces: {stats['total']}")
    print(f"   Successful exploits: {stats['successful_exploits']}")
    print(f"   Failed attempts: {stats['failed_attempts']}")
    print(f"\n   Traces by realm:")
    for realm, count in sorted(stats['by_realm'].items()):
        print(f"     {realm}: {count}")
    print(f"\n   Traces by CWE:")
    for cwe, count in sorted(stats['by_cwe'].items())[:10]:
        print(f"     {cwe}: {count}")
else:
    print("⚠️  No traces loaded. Run Yggdrasil first to generate data!")

In [ ]:
# Visualize dataset composition
if traces and stats:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Exploit success distribution
    ax1.pie([stats['successful_exploits'], stats['failed_attempts']], 
            labels=['Successful', 'Failed'],
            autopct='%1.1f%%',
            colors=['#ff6b6b', '#4ecdc4'])
    ax1.set_title('Exploit Success Rate')
    
    # Top realms
    if stats['by_realm']:
        realms = list(stats['by_realm'].keys())[:10]
        counts = [stats['by_realm'][r] for r in realms]
        ax2.barh(realms, counts, color='#95e1d3')
        ax2.set_xlabel('Trace Count')
        ax2.set_title('Traces by Realm (Top 10)')
    
    plt.tight_layout()
    plt.show()

## Step 3: Prepare Training Data

Split data into training and validation sets.

In [ ]:
def split_dataset(traces, train_ratio=0.8):
    """Split traces into train and validation sets"""
    np.random.shuffle(traces)
    split_idx = int(len(traces) * train_ratio)
    return traces[:split_idx], traces[split_idx:]

def save_jsonl(traces, filename):
    """Save traces to JSONL file"""
    with open(filename, 'w') as f:
        for trace in traces:
            f.write(json.dumps(trace) + '\n')
    print(f"💾 Saved {len(traces)} traces to {filename}")

if traces:
    train_traces, val_traces = split_dataset(traces)
    
    save_jsonl(train_traces, 'train.jsonl')
    save_jsonl(val_traces, 'validation.jsonl')
    
    print(f"\n📊 Dataset split:")
    print(f"   Training: {len(train_traces)} traces (80%)")
    print(f"   Validation: {len(val_traces)} traces (20%)")
else:
    print("⚠️  No traces to split")

## Step 4: Validate Data Format

Ensure data is compatible with OpenAI fine-tuning API.

In [ ]:
def validate_openai_format(traces, sample_size=5):
    """Validate traces match OpenAI format requirements"""
    errors = []
    
    for i, trace in enumerate(traces[:sample_size]):
        # Check required fields
        if 'messages' not in trace:
            errors.append(f"Trace {i}: Missing 'messages' field")
            continue
        
        # Check message structure
        for j, msg in enumerate(trace['messages']):
            if 'role' not in msg or 'content' not in msg:
                errors.append(f"Trace {i}, Message {j}: Missing 'role' or 'content'")
            
            if msg['role'] not in ['system', 'user', 'assistant']:
                errors.append(f"Trace {i}, Message {j}: Invalid role '{msg['role']}'")
    
    if errors:
        print("❌ Validation errors:")
        for error in errors:
            print(f"   {error}")
        return False
    else:
        print("✅ Format validation passed")
        return True

if traces:
    validate_openai_format(train_traces)
    
    # Show example trace
    print("\n📄 Example trace:")
    print(json.dumps(train_traces[0], indent=2))

## Step 5: Estimate Token Usage

Calculate token count for cost estimation.

In [ ]:
def count_tokens(traces, model="gpt-3.5-turbo"):
    """Count tokens in dataset"""
    encoding = tiktoken.encoding_for_model(model)
    total_tokens = 0
    
    for trace in traces:
        for message in trace.get('messages', []):
            total_tokens += len(encoding.encode(message['content']))
    
    return total_tokens

if traces:
    train_tokens = count_tokens(train_traces)
    val_tokens = count_tokens(val_traces)
    
    print(f"🔢 Token count:")
    print(f"   Training: {train_tokens:,} tokens")
    print(f"   Validation: {val_tokens:,} tokens")
    print(f"   Total: {train_tokens + val_tokens:,} tokens")
    
    # Cost estimation (approximate)
    cost_per_1k_tokens = 0.008  # Training cost for gpt-3.5-turbo
    estimated_cost = (train_tokens / 1000) * cost_per_1k_tokens
    print(f"\n💵 Estimated training cost: ${estimated_cost:.2f}")
    print(f"   (Note: This is approximate. Check OpenAI pricing for current rates)")

## Step 6: Upload Training Data

Upload files to OpenAI for fine-tuning.

In [ ]:
# Upload training file
print("📤 Uploading training file...")
training_file = client.files.create(
    file=open('train.jsonl', 'rb'),
    purpose='fine-tune'
)
print(f"✅ Training file uploaded: {training_file.id}")

# Upload validation file
print("\n📤 Uploading validation file...")
validation_file = client.files.create(
    file=open('validation.jsonl', 'rb'),
    purpose='fine-tune'
)
print(f"✅ Validation file uploaded: {validation_file.id}")

## Step 7: Create Fine-Tuning Job

Start the fine-tuning process.

In [ ]:
print("🚀 Creating fine-tuning job...")

fine_tune_job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    validation_file=validation_file.id,
    model="gpt-3.5-turbo",
    suffix="yggdrasil-security"
)

print(f"✅ Fine-tuning job created: {fine_tune_job.id}")
print(f"   Status: {fine_tune_job.status}")
print(f"\n⏳ This may take 10-60 minutes depending on dataset size.")
print(f"\n💡 Monitor progress:")
print(f"   Job ID: {fine_tune_job.id}")
print(f"   Check status in OpenAI dashboard or run the next cell")

## Step 8: Monitor Training Progress

In [ ]:
# Check job status
job_id = fine_tune_job.id  # Or paste your job ID here

job_status = client.fine_tuning.jobs.retrieve(job_id)
print(f"📊 Job Status: {job_status.status}")
print(f"   Model: {job_status.model}")
print(f"   Created: {job_status.created_at}")

if job_status.finished_at:
    print(f"   Finished: {job_status.finished_at}")
    print(f"\n✅ Fine-tuned model ready: {job_status.fine_tuned_model}")
else:
    print(f"\n⏳ Still training... Check back in a few minutes")

## Step 9: Test Fine-Tuned Model

Once training is complete, test the model.

In [ ]:
# Replace with your fine-tuned model ID
FINE_TUNED_MODEL = job_status.fine_tuned_model

def test_model(model_id, prompt):
    """Test fine-tuned model with a security prompt"""
    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": "Security analyst identifying vulnerabilities"},
            {"role": "user", "content": prompt}
        ],
        max_tokens=200
    )
    return response.choices[0].message.content

# Test cases
test_prompts = [
    "Request: POST /api/regulate {pressure: 15000}",
    "Login attempt with username: admin' OR '1'='1'--",
    "User accessing /api/employees/999 without authorization"
]

if FINE_TUNED_MODEL:
    print("🧪 Testing fine-tuned model:\n")
    for i, prompt in enumerate(test_prompts, 1):
        print(f"Test {i}: {prompt}")
        response = test_model(FINE_TUNED_MODEL, prompt)
        print(f"Response: {response}\n")
else:
    print("⚠️  Model not yet available. Wait for training to complete.")

## Step 10: Compare with Base Model

Compare performance against the base model.

In [ ]:
test_prompt = "Request: POST /api/search?q=' UNION SELECT * FROM secrets--"

print("📊 Comparison Test:\n")
print(f"Prompt: {test_prompt}\n")

# Base model
print("🔵 Base Model (gpt-3.5-turbo):")
base_response = test_model("gpt-3.5-turbo", test_prompt)
print(f"{base_response}\n")

# Fine-tuned model
if FINE_TUNED_MODEL:
    print("🟢 Fine-Tuned Model (Yggdrasil-trained):")
    tuned_response = test_model(FINE_TUNED_MODEL, test_prompt)
    print(f"{tuned_response}\n")
    
    print("💡 The fine-tuned model should provide more specific security insights!")

## Summary

✅ **Completed Steps:**
1. Loaded Yggdrasil attack traces
2. Analyzed dataset composition
3. Split into training/validation sets
4. Validated OpenAI format
5. Estimated token usage and costs
6. Uploaded training data
7. Created fine-tuning job
8. Monitored training progress
9. Tested fine-tuned model
10. Compared with base model

## Next Steps

- Integrate model into security workflows
- Evaluate on unseen CTF challenges
- Continuously retrain with new exploit data
- Experiment with different model sizes (gpt-4, etc.)

## Resources

- [OpenAI Fine-Tuning Guide](https://platform.openai.com/docs/guides/fine-tuning)
- [Yggdrasil AI Training Docs](../../docs/AI-TRAINING.md)
- [Attack Trace Format Spec](../../docs/AI-TRAINING.md#what-are-attack-traces)